# Pertemuan 9 — Representasi Pengetahuan: Logika dan Basis Pengetahuan
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia**

Sebelum UTS, agen kita memecahkan masalah dengan **mencari**. Mulai pertemuan ini agen memecahkan masalah dengan **mengetahui**: ia menyimpan pengetahuan tentang dunianya, lalu menarik kesimpulan baru dari pengetahuan itu.

**Setelah menyelesaikan notebook ini Anda mampu:**
1. menulis kalimat logika proposisional dan memeriksa nilai kebenarannya dengan tabel kebenaran;
2. menguji *entailment* (apakah basis pengetahuan menjamin sebuah kesimpulan);
3. merepresentasikan fakta, relasi, dan aturan ber-variabel ala logika orde pertama;
4. membangun ontologi sederhana dan memanfaatkan pewarisan sifat.

Notebook ini adalah **tahap representasi pengetahuan** untuk asesmen A6 (mini-project). Basis pengetahuan yang Anda susun di bagian Latihan akan dipakai lagi pada pertemuan 10 dan 11.

In [ ]:
NAMA = "tulis nama Anda"
NIM = "tulis NIM Anda"
print(f"Notebook P9 dikerjakan oleh: {NAMA} ({NIM})")

import itertools
import pandas as pd, networkx as nx, matplotlib.pyplot as plt

## 1. Logika proposisional

**Proposisi** adalah pernyataan yang bernilai benar atau salah. Proposisi digabung dengan lima penghubung:

| Simbol | Nama | Dibaca | Di notebook ini |
|---|---|---|---|
| ¬P | negasi | tidak P | `("tidak", P)` |
| P ∧ Q | konjungsi | P dan Q | `("dan", P, Q)` |
| P ∨ Q | disjungsi | P atau Q | `("atau", P, Q)` |
| P → Q | implikasi | jika P maka Q | `("maka", P, Q)` |
| P ↔ Q | biimplikasi | P jika dan hanya jika Q | `("setara", P, Q)` |

Kalimat logika ditulis sebagai tuple bersarang. Sebuah **model** adalah pemberian nilai benar/salah untuk setiap simbol.

In [ ]:
def simbol(kalimat):
    """Mengumpulkan semua simbol proposisi dalam sebuah kalimat."""
    if isinstance(kalimat, str):
        return {kalimat}
    return set().union(*(simbol(bagian) for bagian in kalimat[1:]))

def nilai(kalimat, model):
    """Menghitung nilai kebenaran kalimat pada sebuah model (dict simbol -> bool)."""
    if isinstance(kalimat, str):
        return model[kalimat]
    operator, *arg = kalimat
    if operator == "tidak":  return not nilai(arg[0], model)
    if operator == "dan":    return all(nilai(a, model) for a in arg)
    if operator == "atau":   return any(nilai(a, model) for a in arg)
    if operator == "maka":   return (not nilai(arg[0], model)) or nilai(arg[1], model)
    if operator == "setara": return nilai(arg[0], model) == nilai(arg[1], model)
    raise ValueError(f"Operator tidak dikenal: {operator}")

def tulis(kalimat):
    """Mengubah kalimat menjadi notasi logika yang mudah dibaca."""
    if isinstance(kalimat, str): return kalimat
    operator, *arg = kalimat
    if operator == "tidak": return "¬" + tulis(arg[0])
    tanda = {"dan": " ∧ ", "atau": " ∨ ", "maka": " → ", "setara": " ↔ "}[operator]
    return "(" + tanda.join(tulis(a) for a in arg) + ")"

def tabel_kebenaran(*daftar_kalimat):
    sim = sorted(set().union(*(simbol(k) for k in daftar_kalimat)))
    baris = []
    for kombinasi in itertools.product([True, False], repeat=len(sim)):
        model = dict(zip(sim, kombinasi))
        baris.append({**model, **{tulis(k): nilai(k, model) for k in daftar_kalimat}})
    return pd.DataFrame(baris).replace({True: "B", False: "S"})

print(tabel_kebenaran(("maka", "P", "Q"), ("atau", ("tidak", "P"), "Q")).to_string(index=False))

Kedua kolom terakhir identik: `P → Q` **ekuivalen** dengan `¬P ∨ Q`. Perhatikan juga baris ketika P salah: implikasi bernilai benar. "Jika hujan maka jalan basah" tidak dilanggar pada hari yang tidak hujan.

### Tiga istilah penting
- **Valid** (tautologi): benar pada semua model.
- **Satisfiable**: benar pada setidaknya satu model.
- **Kontradiksi**: salah pada semua model.

In [ ]:
def klasifikasi(kalimat):
    sim = sorted(simbol(kalimat))
    hasil = [nilai(kalimat, dict(zip(sim, k))) for k in itertools.product([True, False], repeat=len(sim))]
    if all(hasil): return "valid (tautologi)"
    if any(hasil): return "satisfiable"
    return "kontradiksi"

for k in [("atau", "P", ("tidak", "P")),
          ("dan", "P", ("tidak", "P")),
          ("maka", ("dan", "P", ("maka", "P", "Q")), "Q"),          # modus ponens
          ("maka", ("dan", "Q", ("maka", "P", "Q")), "P")]:         # kekeliruan: menegaskan konsekuen
    print(f"{tulis(k):32s} → {klasifikasi(k)}")

Baris ketiga adalah **modus ponens**: dari `P` dan `P → Q` kita sah menyimpulkan `Q`. Baris keempat adalah kekeliruan yang sering terjadi: dari `Q` dan `P → Q` kita **tidak** boleh menyimpulkan `P`. Jalan basah tidak membuktikan bahwa hujan turun.

## 2. Basis pengetahuan dan entailment

**Basis pengetahuan (KB)** adalah kumpulan kalimat yang dianggap benar oleh agen. Pertanyaan intinya:

> Apakah KB **menjamin** kalimat α? Ditulis `KB ⊨ α`.

Definisinya: `KB ⊨ α` bila α benar pada **setiap** model yang membuat KB benar. Cara paling lugas untuk memeriksanya adalah **model checking**: enumerasi semua model.

Domain contoh kita untuk tiga pertemuan ke depan: **diagnosis gangguan laptop**.

| Simbol | Arti |
|---|---|
| `tidak_menyala` | laptop tidak menyala saat tombol daya ditekan |
| `lampu_daya_mati` | lampu indikator daya tidak menyala |
| `lampu_charger_mati` | lampu pada adaptor/charger tidak menyala |
| `masalah_daya` | gangguan berada pada jalur daya |
| `adaptor_rusak` | adaptor perlu diganti |

In [ ]:
KB = [
    ("maka", ("dan", "tidak_menyala", "lampu_daya_mati"), "masalah_daya"),          # aturan 1
    ("maka", ("dan", "masalah_daya", "lampu_charger_mati"), "adaptor_rusak"),       # aturan 2
    "tidak_menyala",                                                                 # fakta hasil pengamatan
    "lampu_daya_mati",
    "lampu_charger_mati",
]
for k in KB: print("•", tulis(k))

In [ ]:
def entail(kb, alpha, rinci=False):
    """Model checking: True bila alpha benar pada semua model yang memenuhi seluruh kalimat KB."""
    sim = sorted(set().union(simbol(alpha), *(simbol(k) for k in kb)))
    model_kb = 0
    for kombinasi in itertools.product([True, False], repeat=len(sim)):
        model = dict(zip(sim, kombinasi))
        if all(nilai(k, model) for k in kb):
            model_kb += 1
            if not nilai(alpha, model):
                if rinci: print("  model penyangkal:", {s: v for s, v in model.items()})
                return False
    if rinci: print(f"  {2 ** len(sim)} model diperiksa, {model_kb} model memenuhi KB")
    return True

print("KB ⊨ masalah_daya ?", entail(KB, "masalah_daya", rinci=True))
print("KB ⊨ adaptor_rusak?", entail(KB, "adaptor_rusak", rinci=True))

In [ ]:
# Bagaimana bila salah satu fakta belum diketahui?
KB_kurang = [k for k in KB if k != "lampu_charger_mati"]
print("Tanpa fakta lampu_charger_mati:")
print("KB ⊨ adaptor_rusak     ?", entail(KB_kurang, "adaptor_rusak", rinci=True))
print("KB ⊨ ¬adaptor_rusak    ?", entail(KB_kurang, ("tidak", "adaptor_rusak")))

**Perhatikan hasil terakhir.** KB tidak menjamin `adaptor_rusak`, tetapi juga tidak menjamin `¬adaptor_rusak`. Agen yang jujur harus menjawab **"belum diketahui"**, bukan "tidak". Membedakan *salah* dari *tidak diketahui* adalah bagian dari AI yang bertanggung jawab.

**Keterbatasan model checking:** dengan n simbol terdapat 2ⁿ model. Untuk 30 simbol sudah lebih dari satu miliar. Minggu depan kita memakai **inferensi berbasis aturan** (forward dan backward chaining) yang jauh lebih efisien.

## 3. Dari proposisi ke logika orde pertama (FOL)

Logika proposisional tidak bisa "membuka" isi pernyataan. Untuk menyatakan *setiap mahasiswa yang lulus prasyarat boleh mengambil mata kuliah lanjutannya*, kita harus menulis satu aturan per mahasiswa per mata kuliah.

**FOL** menambahkan:
- **objek** (konstanta): `Aisyah`, `PIF403`
- **predikat** (relasi/sifat): `Mahasiswa(Aisyah)`, `Lulus(Aisyah, Algoritma)`
- **variabel** dan **kuantor**: ∀x (untuk semua), ∃x (terdapat)

Contoh aturan:
`∀x ∀m ∀p  Mahasiswa(x) ∧ Prasyarat(p, m) ∧ Lulus(x, p) → BolehAmbil(x, m)`

Di kode, fakta disimpan sebagai tuple `(predikat, argumen...)`. Variabel ditulis dengan awalan `?`.

In [ ]:
FAKTA = {
    ("Mahasiswa", "Aisyah"), ("Mahasiswa", "Bima"), ("Mahasiswa", "Citra"),
    ("MataKuliah", "Algoritma"), ("MataKuliah", "StrukturData"), ("MataKuliah", "KecerdasanBuatan"),
    ("MataKuliah", "PembelajaranMesin"),
    ("Prasyarat", "Algoritma", "StrukturData"),              # Algoritma adalah prasyarat StrukturData
    ("Prasyarat", "StrukturData", "KecerdasanBuatan"),
    ("Prasyarat", "KecerdasanBuatan", "PembelajaranMesin"),
    ("Lulus", "Aisyah", "Algoritma"), ("Lulus", "Aisyah", "StrukturData"),
    ("Lulus", "Bima", "Algoritma"),
}

def cocokkan(pola, fakta, ikatan):
    """Mencocokkan satu pola (boleh ber-variabel) dengan satu fakta. Mengembalikan ikatan variabel baru atau None."""
    if len(pola) != len(fakta): return None
    ikatan = dict(ikatan)
    for p, f in zip(pola, fakta):
        if isinstance(p, str) and p.startswith("?"):
            if ikatan.get(p, f) != f: return None
            ikatan[p] = f
        elif p != f:
            return None
    return ikatan

def tanya(daftar_pola, fakta, ikatan=None):
    """Mencari semua ikatan variabel yang memenuhi SELURUH pola (konjungsi)."""
    ikatan = ikatan or {}
    if not daftar_pola:
        yield ikatan; return
    for f in fakta:
        baru = cocokkan(daftar_pola[0], f, ikatan)
        if baru is not None:
            yield from tanya(daftar_pola[1:], fakta, baru)

print("Siapa saja yang lulus Algoritma?")
for jawab in tanya([("Lulus", "?x", "Algoritma")], FAKTA): print("  ", jawab)

print("Mata kuliah apa yang prasyaratnya sudah dilulusi Aisyah?")
for jawab in tanya([("Lulus", "Aisyah", "?p"), ("Prasyarat", "?p", "?m")], FAKTA): print("  ", jawab)

In [ ]:
ATURAN = [
    # (nama, premis, kesimpulan)
    ("boleh-ambil",
     [("Mahasiswa", "?x"), ("Prasyarat", "?p", "?m"), ("Lulus", "?x", "?p")],
     ("BolehAmbil", "?x", "?m")),
    ("prasyarat-transitif",
     [("Prasyarat", "?a", "?b"), ("Prasyarat", "?b", "?c")],
     ("PrasyaratTakLangsung", "?a", "?c")),
]

def isi(pola, ikatan):
    return tuple(ikatan.get(p, p) for p in pola)

def terapkan_aturan(fakta, daftar_aturan):
    """Menerapkan semua aturan satu putaran dan mengembalikan fakta baru yang diturunkan."""
    baru = set()
    for nama, premis, kesimpulan in daftar_aturan:
        for ikatan in tanya(premis, fakta):
            f = isi(kesimpulan, ikatan)
            if f not in fakta: baru.add(f)
    return baru

turunan = terapkan_aturan(FAKTA, ATURAN)
for f in sorted(turunan): print(f)

Satu aturan ber-variabel menggantikan banyak aturan proposisional. Bima belum boleh mengambil Kecerdasan Buatan karena `Lulus(Bima, StrukturData)` tidak ada di basis pengetahuan.

> **Asumsi dunia tertutup** (*closed-world assumption*): sistem ini menganggap fakta yang tidak tercatat sebagai salah. Asumsi ini praktis, tetapi berisiko: bila nilai Bima belum diinput, sistem akan menolaknya padahal ia sebenarnya lulus. Asumsi seperti ini wajib ditulis dalam dokumentasi sistem.

## 4. Ontologi sederhana

**Ontologi** menata konsep-konsep sebuah domain beserta hubungannya. Dua relasi yang paling umum:
- `adalah` (*is-a*): hubungan subkelas, misalnya *Laptop adalah Komputer*;
- `bagian_dari` (*part-of*): hubungan komposisi, misalnya *Baterai bagian dari Laptop*.

Kekuatan ontologi ada pada **pewarisan**: sifat kelas induk otomatis dimiliki kelas turunannya.

In [ ]:
onto = nx.DiGraph()
adalah = [("Laptop", "Komputer"), ("Desktop", "Komputer"), ("Komputer", "PerangkatElektronik"),
          ("LaptopGaming", "Laptop"), ("Ultrabook", "Laptop"), ("laptop_aisyah", "Ultrabook")]
bagian = [("Baterai", "Laptop"), ("Adaptor", "Laptop"), ("Layar", "Laptop"), ("Prosesor", "Komputer"), ("RAM", "Komputer")]
onto.add_edges_from(adalah, relasi="adalah")
onto.add_edges_from(bagian, relasi="bagian_dari")

SIFAT = {"PerangkatElektronik": {"butuh_listrik": True},
         "Komputer": {"punya_sistem_operasi": True},
         "Laptop": {"portabel": True},
         "LaptopGaming": {"punya_gpu_khusus": True}}

def leluhur(konsep):
    """Semua kelas induk yang dicapai lewat relasi 'adalah' (transitif)."""
    hasil = []
    for _, induk, d in onto.out_edges(konsep, data=True):
        if d["relasi"] == "adalah":
            hasil += [induk] + leluhur(induk)
    return hasil

def apakah(konsep, kelas):
    return kelas == konsep or kelas in leluhur(konsep)

def sifat(konsep):
    """Menggabungkan sifat sendiri dan sifat yang diwarisi dari seluruh leluhur."""
    gabung = {}
    for k in reversed([konsep] + leluhur(konsep)):
        gabung.update(SIFAT.get(k, {}))
    return gabung

def komponen(konsep):
    """Komponen sendiri dan komponen yang diwarisi dari kelas induk."""
    return sorted({bagian for k in [konsep] + leluhur(konsep)
                   for bagian, _, d in onto.in_edges(k, data=True) if d["relasi"] == "bagian_dari"})

print("Leluhur laptop_aisyah            :", leluhur("laptop_aisyah"))
print("laptop_aisyah adalah Komputer?   :", apakah("laptop_aisyah", "Komputer"))
print("laptop_aisyah adalah LaptopGaming?:", apakah("laptop_aisyah", "LaptopGaming"))
print("Sifat laptop_aisyah              :", sifat("laptop_aisyah"))
print("Komponen laptop_aisyah           :", komponen("laptop_aisyah"))

In [ ]:
plt.figure(figsize=(9.5, 5.5))
pos = nx.spring_layout(onto, seed=7, k=1.1)
warna_sisi = ["#0E7C86" if d["relasi"] == "adalah" else "#F2A024" for _, _, d in onto.edges(data=True)]
nx.draw_networkx_nodes(onto, pos, node_size=2100, node_color="#E6F2F1", edgecolors="#12263A")
nx.draw_networkx_labels(onto, pos, font_size=7.5)
nx.draw_networkx_edges(onto, pos, edge_color=warna_sisi, arrows=True, node_size=2100, width=1.8)
plt.plot([], [], color="#0E7C86", label="adalah (is-a)"); plt.plot([], [], color="#F2A024", label="bagian_dari (part-of)")
plt.legend(loc="lower left"); plt.title("Ontologi sederhana domain perangkat komputer"); plt.axis("off"); plt.show()

`laptop_aisyah` tidak pernah dinyatakan butuh listrik atau memiliki RAM. Kedua pengetahuan itu **diturunkan** dari posisi `laptop_aisyah` di dalam ontologi.

---
## Latihan (tahap representasi pengetahuan, formatif menuju A6)

### Latihan 1 — Menerjemahkan kalimat ke logika proposisional
Tuliskan kalimat berikut sebagai tuple, tampilkan dengan `tulis()`, lalu klasifikasikan dengan `klasifikasi()`.
1. "Jika layar gelap dan kipas berputar, maka ada masalah tampilan."
2. "Laptop panas atau kipas kotor, tetapi tidak keduanya."
3. "Jika baterai rusak maka laptop mati mendadak; laptop mati mendadak; jadi baterai rusak." (Apakah penalaran ini valid?)

In [ ]:
# Jawaban Latihan 1

### Latihan 2 — Menguji entailment
Tambahkan ke `KB` dua aturan baru tentang gangguan **tampilan** (misalnya masalah tampilan dan panel layar rusak) beserta fakta pengamatannya. Buktikan dengan `entail()`:
1. satu kesimpulan yang **dijamin** oleh KB;
2. satu kesimpulan yang **tidak dijamin** (tampilkan model penyangkalnya dengan `rinci=True`).

In [ ]:
# Jawaban Latihan 2

### Latihan 3 — Basis pengetahuan domain Anda sendiri
Pilih satu domain untuk kelompok mini-project Anda (contoh: rekomendasi mata kuliah pilihan, diagnosis gangguan jaringan Wi-Fi, penentuan kelayakan peminjaman ruang, identifikasi hama tanaman, pemilihan menu sesuai pantangan makanan). Domain ini dipakai terus sampai pertemuan 11.

Susun:
1. **minimal 12 fakta** dengan sedikitnya 3 predikat berbeda (gunakan format tuple seperti `FAKTA`);
2. **minimal 4 aturan ber-variabel** (format seperti `ATURAN`), lalu jalankan `terapkan_aturan` dan tampilkan fakta turunannya;
3. **ontologi** dengan minimal 8 konsep, memakai relasi `adalah` dan `bagian_dari`, serta minimal 3 sifat yang diwariskan;
4. daftar **asumsi** yang Anda buat (misalnya asumsi dunia tertutup) dan **sumber pengetahuan** yang dipakai.

In [ ]:
# Jawaban Latihan 3: fakta dan aturan

In [ ]:
# Jawaban Latihan 3: ontologi

*Asumsi dan sumber pengetahuan:*

### Refleksi
1. Apa perbedaan antara "KB menyatakan α salah" dan "KB tidak menjamin α"? Mengapa perbedaan ini penting bagi pengguna sistem?
2. Sebutkan satu pengetahuan pada domain Anda yang **sulit** dinyatakan dengan benar/salah. Bagaimana rencana Anda menanganinya? (Petunjuk: pertemuan 11.)
3. Siapa yang berhak memutuskan isi basis pengetahuan pada domain Anda, dan apa risikonya bila pengetahuan itu keliru atau usang?

*Jawaban:*

---
**Pengumpulan:** `P09_NIM_Nama.ipynb` diunggah ke LMS sebagai bukti tahap representasi pengetahuan.